In [24]:
from pymongo import MongoClient
from datetime import datetime

client = MongoClient("mongodb://localhost:27017/")
db = client["jotuns_lair"]

rooms    = db["rooms"]
users    = db["users"]
loot     = db["loot"]
monsters = db["monsters"]

### A. Consulta para extraer los comentarios y exportación a JSON

Actualmente los comentarios están almacenados como arrays embebidos en las colecciones `Rooms` y `Users`. Para centralizarlos en una única colección, necesitamos desanidar esos arrays y formatear los campos según la nueva estructura de la colección `Hints`.

Dado que un comentario pertenece inherentemente a una sala y dentro de él ya se incluye la información del usuario (subdocumento `publish_by`), la forma más sencilla y sin duplicados es **extraer todos los comentarios únicamente desde la colección `Rooms`**, que contiene la relación completa habitación‑usuario. Esto evita posibles discrepancias entre ambas colecciones.

**Pipeline de agregación en MongoDB** (para ejecutar en `mongosh` o en la sección *Aggregation* de Compass):

```javascript
mongodb.rooms.aggregate([
  {
    $unwind: "$hints"   // Separa cada comentario del array
  },
  {
    $project: {
      _id: 0,            // Se genera un nuevo _id automáticamente al exportar
      Creation_date: "$hints.creation_date",
      HintText: "$hints.hintText",
      Category: "$hints.category",
      References_room: {
        IdR: "$room_id",
        Name: "$room_name",
        IdD: "$dungeon_id",
        Dungeon: "$dungeon_name"
      },
      Publish_by: {
        Email: "$hints.publish_by.email",
        User_name: "$hints.publish_by.user_name",
        CreationDate: "$hints.publish_by.creation_date",
        Country: "$hints.publish_by.country"
      }
    }
  }
]);
```

**Exportación del resultado a JSON:**

- **Opción 1 (MongoDB Compass):** tras ejecutar la agregación, hacer clic en *Export Data* y elegir formato JSON.
- **Opción 2 (script personalizado):** escribir un script en Python (usando `pymongo`) que devuelva los resultados y los escriba en un archivo JSON.

El archivo `hints.json` resultante contendrá los documentos con la estructura deseada.


### B. Creación de la colección `Hints`, importación y limpieza de las colecciones originales

1. **Crear la colección `Hints` e importar el JSON**  
   Desde la terminal:
   ```bash
   mongoimport --db mongodb --collection Hints --file hints.json --jsonArray
   ```
   Esto inserta todos los comentarios extraídos en la nueva colección. Se presupone que el nombre del archivo es `hints.json` y que el formato es un array de documentos JSON y que la base de datos se llama `mongodb`.

   También se puede usar MongoDB Compass para importar el archivo JSON directamente a la colección `Hints` (creando una nueva colección e importando el JSON usando Add Data o directamente Import Data).

2. **Eliminar el campo `hints` de las colecciones `Rooms` y `Users`**  
   Una vez verificada la correcta importación, se eliminan los arrays embebidos:

   ```javascript
   mongodb.rooms.updateMany({}, { $unset: { hints: "" } });
   mongodb.users.updateMany({}, { $unset: { hints: "" } });
   ```

   > **Nota 1:** También podríamos eliminar el campo `hints` de los usuarios si existiera, ya que ahora los comentarios residen en `Hints` y se referencian por email del usuario.
   
   > **Nota 2:** Podríamos también haber usado $out y $merge para crear la colección `Hints` directamente.




In [27]:
def eliminar_hints():

    eliminar_rooms = rooms.update_many({}, {"$unset": {"hints": ""}})
    eliminar_users = users.update_many({}, {"$unset": {"hints": ""}})

    return eliminar_rooms, eliminar_users

In [28]:
eliminar_hints()

(UpdateResult({'n': 718, 'nModified': 0, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True),
 UpdateResult({'n': 9838, 'nModified': 0, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True))

### C. Adaptación de los endpoints al nuevo esquema

Ahora que los comentarios están en una colección independiente, las operaciones de lectura y escritura deben dirigirse a `Hints`. Se describen los cambios funcionales para cada endpoint.


#### `POST /comment`

Este endpoint añade un nuevo comentario. Recibe como parámetros: user_email (str), room_id (int), text (str), category (str).

*Antes:* Insertaba el comentario en el array `hints` de la sala y  también en el usuario.

*Ahora:*
- Inserta un único documento en `Hints` con los datos recibidos (`user_email`, `room_id`, `text`, `category`) y añade:
  - La fecha actual como `Creation_date`.
  - El subdocumento `References_room` rellenado consultando la sala (id, nombre, id de mazmorra, nombre de mazmorra).
  - El subdocumento `Publish_by` obtenido de la colección `Users` a partir del `user_email` (nombre, país, fecha de creación del usuario).
- Si se aplicó el **patrón Computed** (ejercicio anterior), se actualiza además los contadores de categorías en la sala correspondiente (no lo haremos por no haberlo implementado y ser ajeno a la práctica).


In [20]:
hints = db["hints"]

In [21]:
def post_comment(user_email, room_id, text, category):
    user = users.find_one({"email": user_email})
    room = rooms.find_one({"room_id": room_id})
    new_comment = {
        "Creation_date": datetime.utcnow().isoformat(),
        "HintText": text,
        "Category": category,
        "References_room": {
            "IdR": room["room_id"],
            "Name": room["room_name"],
            "IdD": room["dungeon_id"],
            "Dungeon": room["dungeon_name"]
        },
        "Publish_by": {
            "Email": user_email,
            "User_name": user["user_name"],
            "CreationDate": user["creation_date"],
            "Country": user["country"]
        }
    }
    hints.insert_one(new_comment)

    """
    Como no hemos realizado los patrones de diseño, lo dejamos aquí únicamente como comentario. 
    db.Rooms.update_one(
        {"room_id": room_id},
        {"$inc": {f"comments_by_category.{category}": 1}}
    )
    """

In [8]:
post_comment("adanherranz@example.org", 2, "Este es un comentario de ejemplo para la habitación 1 en la categoría 'General'.", "General")

Comprobamos que funciona correctamente. 

In [9]:
def get_comment(email, room_id, category=None):
    query = {
        "Publish_by.Email": email,
        "References_room.IdR": room_id
    }
    if category:
        query["Category"] = category
    return list(hints.find(query))

In [10]:
get_comment("adanherranz@example.org", 2, "General")

[{'_id': ObjectId('6a0f47859c7a6140dc760cde'),
  'Creation_date': '2026-05-21T17:57:25.299092',
  'HintText': "Este es un comentario de ejemplo para la habitación 1 en la categoría 'General'.",
  'Category': 'General',
  'References_room': {'IdR': 2,
   'Name': 'sanctuary ',
   'IdD': 0,
   'Dungeon': 'Burghap, Prison of the Jealous Hippies'},
  'Publish_by': {'Email': 'adanherranz@example.org',
   'User_name': 'auroraespana',
   'CreationDate': '2022-01-02',
   'Country': 'es_ES'}},
 {'_id': ObjectId('6a0f4d294183adae3da5b447'),
  'Creation_date': '2026-05-21T18:21:29.620788',
  'HintText': "Este es un comentario de ejemplo para la habitación 1 en la categoría 'General'.",
  'Category': 'General',
  'References_room': {'IdR': 2,
   'Name': 'sanctuary ',
   'IdD': 0,
   'Dungeon': 'Burghap, Prison of the Jealous Hippies'},
  'Publish_by': {'Email': 'adanherranz@example.org',
   'User_name': 'auroraespana',
   'CreationDate': '2022-01-02',
   'Country': 'es_ES'}}]

#### `GET /room/{room_id}`

Este endpoint recibe el **id de una habitación** y devuelve la siguiente información:

- **`idR`**
- **`name`**
- **`inWP`**
- **`outWP`**

Además, incluye:

- El **número de monstruos de cada tipo** presentes en la habitación.
- El **total de oro** que valen los tesoros de la sala.
- Los **últimos 20 comentarios** realizados sobre esa habitación.

Cada comentario debe incluir:

- **`userName`**
- **`country`**
- **`creationDate`** del usuario que lo realizó
- **Texto**
- **Fecha de publicación**
- **Categoría** del comentario


*Antes:* Devolvía directamente la sala con su array `hints` embebido y extraía los últimos 20 comentarios.

*Ahora:*
- La consulta principal a `Rooms` ya no contiene los comentarios.
- Hay que realizar una **segunda consulta** a `Hints`, filtrando por `References_room.IdR == room_id`, ordenando por `Creation_date` descendente y limitando a 20.
- Si se desea evitar queries adicionales, se podría emplear un **`$lookup`** desde `Rooms` hacia `Hints`, pero dado que el endpoint solo requiere 20 comentarios, una query separada con un índice en `(References_room.IdR, Creation_date)` es muy eficiente.

En nuestro caso, usaremos un `$lookup` para obtener los comentarios junto con la información de la sala en una sola consulta, aunque también se podría hacer con dos consultas separadas.

In [11]:
hints = db["Hints"]

In [39]:
def get_room(room_id: int):
    """
    GET /room/{room_id}
    Devuelve todos los campos de la habitación más los 20 últimos comentarios
    con texto, fecha, categoría, id/nombre de usuario y id/nombre de dungeon.
    """
    pipeline = [
        # Filtrar por room_id
        {"$match": {"room_id": room_id}},

        {
            "$lookup": {
                "from": "Hints", 
                "let": {"id_actual": "$_id"},
                "pipeline": [
                    {"$unwind": "$References_room"},
                    
                    {"$match": {
                        "References_room.IdR": room_id 
                    }},
                    
                    {"$sort": {
                        "Creation_date": -1
                    }},
                    
                    {"$limit": 20},
                    
                    {"$project": {
                        "_id": 0,
                        "Creation_date": 1,
                        "HintText": 1,
                        "Category": 1,
                        "Publish_by": {
                            "User_name": 1,
                            "Country": 1,
                            "CreationDate": 1
                        }
                    }}
                ],
                "as": "hint"
            }
        },

        {
            "$lookup": {
                "from": "monsters",
                "let": {"id_actual": "$_id"},
                "pipeline": [
                    {"$unwind": "$in_rooms"},
                    {"$match": {"in_rooms.room_id": room_id}},
                    
                    {"$group": {
                        "_id": "$type",
                        "count": {"$sum": "$in_rooms.amount"}
                    }},
                    
                    {"$project": {
                        "_id": 0,
                        "type": "$_id",
                        "count": 1
                    }}
                ],
                "as": "monsters_info"
            }
        },

        {"$project": {
            "_id": 0,
            "room_id": 1,
            "room_name": 1,
            "in_waypoint": 1,
            "out_waypoint": 1,
            "monsters_info": 1,
            "treasure_tot_value": 1,
            "hint": 1 
        }}
    ]

    result = list(rooms.aggregate(pipeline))
    return result[0] if result else None

In [40]:
get_room(196)

{'room_id': 196,
 'room_name': 'throne room of ninjas',
 'in_waypoint': None,
 'out_waypoint': None,
 'hint': [{'Creation_date': '2022-09-24 10:39:09.000000',
   'HintText': '音樂數據服務根據實現的是准備他們.過程隻要行業的話孩子經驗網上.\\n學校大小教育點擊起來無法之間.成為功能更多.\\n安全價格還有人民閱讀.簡介美國的是教育市場影響.圖片來自以及這個.\\n地方你的最新經驗一定.電話詳細論壇注意他們男人.一些系列經驗注冊深圳發生進行.\\n簡介這是是一控制.音樂軟體密碼已經成為當前.音樂作品客戶不要以后手機孩子.\\n設備全國游戲單位產品.游戲廣告一點由於.更新一點包括還有主題.\\n時間繼續發表是否生活那些.公司正在然后狀態感覺不過.環境其實隻有准備開發汽車她的.\\n其實一下歡迎其中個人隻有如果密碼.他的系列免費發展.\\n發展方法重要.看到日期孩子設備語言謝謝推薦一些.\\n精華更多密碼電腦完成閱讀我們.一點網絡雖然.還有會員而且點擊通過一樣來自.\\n成為時候操作覺得首頁也是台灣兩個.聯系東西進行的話研究.\\n非常關於公司到了.他們今年網絡.\\n技術會員城市隻是.我的能夠處理如何看到發布當然.地方經濟國內新聞電子.選擇以及全國責任今年而且經營的人.\\n是否這個名稱建設關系網絡.音樂經驗方面最大一樣任何對於部門.\\n就是資源雖然銷售投資最大圖片如何.對於地方主題質量閱讀可是還是.',
   'Category': 'hint',
   'Publish_by': {'User_name': 'ufeng',
    'CreationDate': '2022-01-14',
    'Country': 'zh_TW'}},
  {'Creation_date': '2022-07-07 06:36:22.000000',
   'HintText': 'Like successful above big after own piece over. Of interesting hear just guess three financial. Detail find bre

#### `GET /dungeon/{dungeon_id}`

Este endpoint recibe el id de una mazmorra y devuelve información sobre una mazmorra del juego. 
Debe devolver: idM, name y lore. Además, este endpoint se utiliza para alimentar un grafo interactivo por lo que requiere la siguiente información: 

1) el nombre e id de cada habitación de la mazmorra; 
2) las conexiones entre habitaciones de la mazmorra; 
3) el id y el nombre de los monstruos que aparecen en cada habitación; 
4) el id y el nombre de los tesoros que aparecen en cada habitación; 
5) El número de comentarios de cada categoría que hay en cada habitación. 

*Antes:* Recorría las habitaciones de la mazmorra y, a partir de sus `hints` embebidos, contaba los comentarios por categoría.

*Ahora:* Existen dos enfoques:

1. **Sin precalcular (más caro):**  
   Se usa un pipeline de agregación que:
   - Filtra las salas de la mazmorra (`$match`).
   - Hace `$lookup` con `Hints` a través de `room_id`.
   - Desanida y agrupa por sala y categoría para contar.
   - Vuelve a agrupar para integrar los resultados en las salas.

2. **Con el patrón Computed:**  
   Si en el endpoint `POST /comment` se mantienen actualizados los contadores en el documento de la sala (campo `comments_by_category`), la obtención de la mazmorra no requiere ningún cambio: los contadores ya están desnormalizados en cada sala. La función simplemente sigue leyendo los documentos de las habitaciones y ya dispone de los números.

Dado que `GET /dungeon` tiene 1M de accesos diarios, la opción 2 es la única viable en producción. Por tanto, la función del endpoint apenas se modifica (salvo la eliminación de la lógica que contaba comentarios sobre arrays, que ya no existe).

In [60]:
def get_dungeon(dungeon_id):
    pipeline = [
        {
            "$match": { "dungeon_id": dungeon_id }
        },
        {
            "$lookup": {
                "from": "Hints",                     
                "localField": "room_id",             
                "foreignField": "References_room.IdR", 
                "as": "room_hints"                   
            }
        },
        {
            "$addFields": {
                "monsters_graph": {
                    "$setUnion": [
                        {
                            "$map": {
                                "input": { "$ifNull": ["$monsters", []] },
                                "as": "m",
                                "in": { "id": "$$m.id", "name": "$$m.name" } 
                            }
                        }
                    ]
                },
                "loot_graph": {
                    "$setUnion": [
                        {
                            "$map": {
                                "input": { "$ifNull": ["$loot", []] },
                                "as": "l",
                                "in": { "id": "$$l.id", "name": "$$l.name" } 
                            }
                        }
                    ]
                },
                "comments_summary": {
                    "$map": {
                        "input": {
                            "$setUnion": [
                                {
                                    "$map": {
                                        "input": { "$ifNull": ["$room_hints", []] },
                                        "as": "h",
                                        "in": "$$h.Category" 
                                    }
                                }
                            ]
                        },
                        "as": "hints_category",
                        "in": {
                            "category": "$$hints_category",
                            "cantidad": {
                                "$size": {
                                    "$filter": {
                                        "input": { "$ifNull": ["$room_hints", []] },
                                        "as": "h",
                                        "cond": { "$eq": ["$$h.Category", "$$hints_category"] }
                                    }
                                }
                            }
                        }
                    }
                }
            }
        },
        {
            "$group": {
                "_id": "$dungeon_id",
                "name": { "$first": "$dungeon_name" },
                "graph_rooms": {
                    "$push": {
                        "room_id": "$room_id", 
                        "room_name": "$room_name",
                        "connections": { "$ifNull": ["$rooms_connected", []] },
                        "monsters": "$monsters_graph",
                        "loot": "$loot_graph",
                        "comments_count": "$comments_summary"
                    }
                }
            }
        },
        {
            "$project": {
                "_id": 0,
                "idD": "$_id", 
                "name": 1,
                "rooms": "$graph_rooms"
            }
        }
    ]

    result = list(db.rooms.aggregate(pipeline))

    return result[0] if result else {}

In [61]:
get_dungeon(1)

{'name': 'Burgstream, Culverts of the Bashful Sumo Wrestlers',
 'idD': 1,
 'rooms': [{'room_id': 33,
   'room_name': 'drawing room of unknowns',
   'connections': [{'room_id': 45, 'room_name': 'great hall of rogues'}],
   'monsters': [],
   'loot': [],
   'comments_count': [{'category': 'bug', 'cantidad': 3},
    {'category': 'hint', 'cantidad': 1},
    {'category': 'lore', 'cantidad': 5}]},
  {'room_id': 34,
   'room_name': 'hall ',
   'connections': [{'room_id': 35,
     'room_name': 'panicky sanctuary of scientists'}],
   'monsters': [],
   'loot': [],
   'comments_count': [{'category': 'bug', 'cantidad': 3},
    {'category': 'hint', 'cantidad': 3},
    {'category': 'lore', 'cantidad': 7},
    {'category': 'suggestion', 'cantidad': 2}]},
  {'room_id': 37,
   'room_name': 'scullery of unknowns',
   'connections': [{'room_id': 36, 'room_name': 'raspy lounge of ninjas'},
    {'room_id': 38, 'room_name': 'parlour '},
    {'room_id': 46, 'room_name': 'grumpy drawing room of otaku'},
    

#### `GET /user/{email}`

Este endpoint recibe el email de un usuario y devuelve todos los campos de un usuario. Además, incluye 
los 20 últimos comentarios que ha realizado ese usuario. De cada comentario incluye, el texto, la fecha 
de creación, la categoría, el id (Room.IdR) y nombre (Room.name) de la habitación a la que hace 
referencia el comentario, el id (Dungeon.IdD) y nombre (Dungeon.name) de la mazmorra donde está la 
habitación. 

*Antes:* Devolvía los últimos 20 comentarios del usuario probablemente desde un array embebido en el documento del usuario.

*Ahora:*
- Se obtiene el usuario (sin array de comentarios).
- Se consulta la colección `Hints` filtrando por `Publish_by.Email`, ordenando por `Creation_date` descendente y limitando a 20.
- Se adjuntan esos comentarios al objeto de usuario devuelto.


In [83]:
def get_user(email: str):
    """
    GET /user/{email}
    Devuelve todos los campos del usuario más los 20 últimos comentarios
    con texto, fecha, categoría, id/nombre de room y id/nombre de dungeon.
    """
    pipeline = [
        # Filtrar por email
        {"$match": {"email": email}},

        {
            "$lookup": {
                "from": "Hints", 
                "let": {"id_actual": "$_id"},
                "pipeline": [
                     	 {"$unwind": "$Publish_by"},
                    
                        {"$match": {
                            "Publish_by.Email": email
                        }},

                        {"$sort": {
                                "Creation_date": -1
                            }},

                        {"$limit": 20},

                    {"$project": {
                        "_id": 0,
                        "HintText": 1,
                        "Creation_date": 1, 
                        "Category": 1,
                        "References_room.IdR": 1,
                        "References_room.Name": 1, 
                        "References_room.IdD": 1,
                        "References_room.Dungeon": 1
                    }}
                ],
                "as": "hint"
            }
        },

        # Proyectar solo los campos necesarios
        {"$project": {
            "_id": 0,
            "email": 1,
            "user_name": 1,
            "creation_date": 1,
            "country": 1,
            "hint": 1
        }}
    ]

    result = list(users.aggregate(pipeline))
    return result[0] if result else None

In [84]:
get_user("lei26@example.org")

{'email': 'lei26@example.org',
 'country': 'zh_CN',
 'user_name': 'nawang',
 'creation_date': '2020-02-02',
 'hint': [{'Creation_date': '2022-07-11 15:28:56.000000',
   'HintText': '介绍威望最后大家全国学校来自电话.美国全国发表会员其他网络系列.\\n东西发表等级分析.应该特别发布.\\n不要设计一些.各种重要一个当前回复今天.女人具有专业个人对于这么.\\n提供来源论坛游戏联系有些之间.目前出现自己回复资料.手机工程只有说明类别然后.\\n通过以后手机这样自己时间.\\n部门大家发展为什到了合作北京.为什一次参加具有必须.各种或者增加个人国内起来.\\n作品因为最后他的发现.谢谢发表今天需要通过.登录目前美国注意分析原因.\\n品牌免费国家软件对于一点.那个作品就是大学.\\n这么是一帮助国际的是点击.社会当然一些我的.能力只有得到组织不过.东西各种成为.\\n这个知道技术女人他的国际.类型欢迎操作本站因为深圳拥有价格.已经虽然这么方式信息她的.\\n大学到了然后人民单位.以及网站学习.游戏一些本站一个一点就是知道对于.\\n企业或者标题系列名称资料.内容自己资源帖子实现.语言她的他的实现使用发展.\\n拥有音乐个人大家经济游戏主题.工程公司手机专业科技.然后就是关于.\\n新闻联系非常资料浏览.工具那个怎么起来投资.评论状态有关经营孩子.\\n计划完全进行一些.其他电子人民发现情况那么以下世界.如果数据他的内容.',
   'Category': 'suggestion',
   'References_room': {'IdR': 2,
    'Name': 'sanctuary ',
    'IdD': 0,
    'Dungeon': 'Burghap, Prison of the Jealous Hippies'}}]}

### Impacto general del cambio

- **Ventajas:**  
  - Los comentarios quedan centralizados y normalizados, facilitando las consultas analíticas (como las del apartado 2) sin tener que recorrer arrays enormes.  
  - Se elimina la duplicación de datos (un mismo comentario ya no está en `Rooms` y en `Users`).  
  - Las operaciones de escritura (`POST`) son atómicas sobre un solo documento, evitando la actualización de arrays que pueden crecer indefinidamente.

- **Desventajas y contramedidas:**  
  - Las lecturas de comentarios ahora requieren consultas adicionales a `Hints`. Esto se mitiga con **índices compuestos** adecuados:  
    - `{References_room.IdR: 1, Creation_date: -1}` para `GET /room`.  
    - `{Publish_by.Email: 1, Creation_date: -1}` para `GET /user`.  
    - `{References_room.IdD: 1, Category: 1}` para agregaciones analíticas por mazmorra.  
  - El endpoint más crítico (`GET /dungeon`) no sufre penalización porque se apoya en los **contadores precalculados** (patrón Computed) que se mantienen durante la inserción de comentarios.
